# 02. 확률과 통계 복습

## 학습 목표
- 확률의 기초 개념과 조건부 확률을 이해하고 시뮬레이션으로 검증
- 주요 확률분포를 시각화하고 ML에서의 활용을 파악
- 베이즈 정리, MLE, 정보이론의 핵심 아이디어를 코드로 구현

## 참고 자료
- [StatQuest - Probability](https://www.youtube.com/watch?v=uzkc-qNVoOk)
- [3Blue1Brown - Bayes' theorem](https://www.youtube.com/watch?v=HZGCoVF3YvM)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 1. 확률 기초

### 1.1 확률이란?

확률은 어떤 사건이 일어날 가능성을 0~1 사이의 숫자로 표현한 것.

$$P(A) = \frac{\text{사건 A가 일어나는 경우의 수}}{\text{전체 경우의 수}}$$

ML에서 확률은 모든 곳에 등장:
- 분류 모델의 출력: "이 이메일이 스팸일 확률 = 0.95"
- 언어 모델: "다음 단어가 'the'일 확률 = 0.12"
- 손실 함수: Cross-entropy는 확률분포 간의 차이를 측정

In [ ]:
# 주사위 시뮬레이션으로 확률 확인
np.random.seed(42)
n_rolls = 100000
rolls = np.random.randint(1, 7, size=n_rolls)

# 각 눈의 확률 (이론값: 1/6 ~ 0.1667)
for face in range(1, 7):
    freq = np.mean(rolls == face)
    print(f"P({face}) = {freq:.4f}  (이론값: {1/6:.4f})")

# 짝수가 나올 확률
p_even = np.mean(rolls % 2 == 0)
print(f"\nP(짝수) = {p_even:.4f}  (이론값: 0.5000)")

### 1.2 조건부 확률 (Conditional Probability)

사건 B가 일어났다는 조건 하에 사건 A가 일어날 확률:

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

**ML에서의 활용**: 분류 모델은 입력 X가 주어졌을 때 클래스 Y의 조건부 확률 $P(Y|X)$를 학습한다.

In [ ]:
# 주사위 조건부 확률 시뮬레이션
# P(6이 나옴 | 짝수가 나옴) = ?
# 이론값: P(6 AND 짝수) / P(짝수) = (1/6) / (3/6) = 1/3

np.random.seed(42)
n = 100000
rolls = np.random.randint(1, 7, size=n)

even_rolls = rolls[rolls % 2 == 0]  # 짝수인 경우만 필터
p_6_given_even = np.mean(even_rolls == 6)

print(f"P(6 | 짝수) = {p_6_given_even:.4f}  (이론값: {1/3:.4f})")
print(f"짝수 중 6의 비율: {np.sum(even_rolls == 6)} / {len(even_rolls)}")

In [ ]:
# 조건부 확률 시각화: 두 주사위의 합
np.random.seed(42)
n = 100000
die1 = np.random.randint(1, 7, size=n)
die2 = np.random.randint(1, 7, size=n)
total = die1 + die2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 전체 합의 분포
ax = axes[0]
ax.hist(total, bins=np.arange(1.5, 13.5), density=True, alpha=0.7, color='steelblue', edgecolor='black')
ax.set_xlabel('Sum of two dice')
ax.set_ylabel('Probability')
ax.set_title('P(Sum)')
ax.set_xticks(range(2, 13))
ax.grid(True, alpha=0.3)

# 오른쪽: die1=6일 때 합의 조건부 분포
ax = axes[1]
total_given_6 = total[die1 == 6]
ax.hist(total_given_6, bins=np.arange(6.5, 13.5), density=True, alpha=0.7, color='coral', edgecolor='black')
ax.set_xlabel('Sum of two dice')
ax.set_ylabel('Probability')
ax.set_title('P(Sum | Die1 = 6)')
ax.set_xticks(range(7, 13))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("첫 번째 주사위가 6이면 합의 분포가 7~12로 제한된다 (균일분포)")

### 1.3 독립 (Independence)

두 사건 A, B가 독립이면: $P(A \cap B) = P(A) \cdot P(B)$

독립이면 한 사건의 발생이 다른 사건에 영향을 주지 않는다.

$$P(A|B) = P(A) \quad \text{(B가 일어나도 A의 확률은 변하지 않음)}$$

In [ ]:
# 독립 사건 시뮬레이션: 동전 2개 던지기
np.random.seed(42)
n = 100000
coin1 = np.random.choice(['H', 'T'], size=n)
coin2 = np.random.choice(['H', 'T'], size=n)

# P(coin1=H) ~ 0.5
p_h1 = np.mean(coin1 == 'H')
# P(coin2=H) ~ 0.5
p_h2 = np.mean(coin2 == 'H')
# P(coin1=H AND coin2=H) ~ 0.25
p_both_h = np.mean((coin1 == 'H') & (coin2 == 'H'))

print(f"P(coin1=H) = {p_h1:.4f}")
print(f"P(coin2=H) = {p_h2:.4f}")
print(f"P(both H)  = {p_both_h:.4f}")
print(f"P(H1) * P(H2) = {p_h1 * p_h2:.4f}")
print(f"\n독립인가? P(A and B) ~ P(A) * P(B)? -> {np.isclose(p_both_h, p_h1 * p_h2, atol=0.01)}")

# 종속 사건 예시: 카드 뽑기 (비복원)
print("\n--- 종속 사건: 카드 비복원 추출 ---")
print(f"P(첫 번째 에이스) = 4/52 = {4/52:.4f}")
print(f"P(두 번째 에이스 | 첫 번째 에이스) = 3/51 = {3/51:.4f}")
print(f"P(두 번째 에이스) = 4/52 = {4/52:.4f}  (조건 없을 때)")
print(f"-> 조건부 확률이 다르므로 종속!")

---
## 2. 확률분포

### 2.1 베르누이 분포 (Bernoulli Distribution)

결과가 성공(1) 또는 실패(0) 두 가지뿐인 시행.

$$P(X=1) = p, \quad P(X=0) = 1-p$$

- 평균: $\mu = p$
- 분산: $\sigma^2 = p(1-p)$

**ML에서**: 이진 분류의 기초. 스팸/정상, 양성/음성 등.

In [ ]:
# 베르누이 분포: 편향 동전 (앞면 확률 0.7)
np.random.seed(42)
p = 0.7
n = 10000
samples = np.random.binomial(1, p, size=n)  # 베르누이 = 이항(n=1, p)

print(f"p = {p}인 베르누이 분포에서 {n}번 시행")
print(f"성공 비율: {np.mean(samples):.4f} (이론값: {p})")
print(f"평균: {np.mean(samples):.4f} (이론값: {p})")
print(f"분산: {np.var(samples):.4f} (이론값: {p*(1-p):.4f})")

# 시각화
fig, ax = plt.subplots(figsize=(6, 4))
values, counts = np.unique(samples, return_counts=True)
ax.bar(values, counts / n, color=['coral', 'steelblue'], edgecolor='black', width=0.4)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Failure (0)', 'Success (1)'])
ax.set_ylabel('Probability')
ax.set_title(f'Bernoulli Distribution (p={p})')
for v, c in zip(values, counts / n):
    ax.text(v, c + 0.01, f'{c:.3f}', ha='center', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.show()

### 2.2 이항분포 (Binomial Distribution)

베르누이 시행을 n번 반복했을 때 성공 횟수의 분포.

$$P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$$

- 평균: $\mu = np$
- 분산: $\sigma^2 = np(1-p)$

In [ ]:
# 이항분포 시뮬레이션 + 시각화
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

params = [(10, 0.5), (20, 0.5), (20, 0.3)]

for ax, (n_trials, p) in zip(axes, params):
    # 시뮬레이션
    samples = np.random.binomial(n_trials, p, size=10000)

    # 이론적 PMF
    k_values = np.arange(0, n_trials + 1)
    pmf = stats.binom.pmf(k_values, n_trials, p)

    ax.hist(samples, bins=np.arange(-0.5, n_trials + 1.5), density=True,
            alpha=0.6, color='steelblue', label='Simulation')
    ax.plot(k_values, pmf, 'ro-', markersize=4, label='Theory (PMF)')
    ax.axvline(n_trials * p, color='green', linestyle='--',
               label=f'Mean = {n_trials*p:.1f}')
    ax.set_xlabel('Number of successes (k)')
    ax.set_ylabel('Probability')
    ax.set_title(f'B({n_trials}, {p})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Binomial Distribution', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# 검증
n_trials, p = 20, 0.5
samples = np.random.binomial(n_trials, p, size=100000)
print(f"B({n_trials}, {p}):")
print(f"  시뮬레이션 평균: {np.mean(samples):.2f} (이론값: {n_trials*p})")
print(f"  시뮬레이션 분산: {np.var(samples):.2f} (이론값: {n_trials*p*(1-p)})")

### 2.3 정규분포 (Normal / Gaussian Distribution)

자연에서 가장 많이 나타나는 분포. ML에서 가장 중요한 연속 분포.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**ML에서의 활용**:
- 가중치 초기화: $W \sim \mathcal{N}(0, \sigma^2)$
- VAE (Variational Autoencoder)의 잠재 공간
- 중심극한정리: 많은 통계량이 정규분포에 수렴

In [ ]:
# 정규분포 시각화: 다양한 평균과 분산
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.linspace(-8, 8, 500)

# 왼쪽: 평균 변화
ax = axes[0]
for mu in [-2, 0, 2]:
    y = stats.norm.pdf(x, mu, 1)
    ax.plot(x, y, linewidth=2, label=f'mu={mu}, sigma=1')
ax.set_title('Effect of Mean (mu)')
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: 분산 변화
ax = axes[1]
for sigma in [0.5, 1, 2]:
    y = stats.norm.pdf(x, 0, sigma)
    ax.plot(x, y, linewidth=2, label=f'mu=0, sigma={sigma}')
ax.set_title('Effect of Std Dev (sigma)')
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 시뮬레이션으로 정규분포 성질 확인
np.random.seed(42)
samples = np.random.normal(loc=0, scale=1, size=100000)

print(f"표본 평균: {np.mean(samples):.4f} (이론값: 0)")
print(f"표본 분산: {np.var(samples):.4f} (이론값: 1)")

# 68-95-99.7 법칙
print("\n--- 68-95-99.7 법칙 ---")
for k in [1, 2, 3]:
    within = np.mean(np.abs(samples) < k) * 100
    expected = [68.27, 95.45, 99.73][k-1]
    print(f"|x| < {k}*sigma: {within:.2f}% (이론값: {expected}%)")

### 2.4 중심극한정리 (Central Limit Theorem)

어떤 분포든 **표본 평균**을 충분히 많이 모으면 정규분포에 가까워진다.

$$\bar{X}_n \xrightarrow{d} \mathcal{N}\left(\mu, \frac{\sigma^2}{n}\right)$$

이것이 정규분포가 ML에서 그토록 중요한 근본적인 이유.

In [ ]:
# 중심극한정리 시각화: 주사위(균일분포)의 표본 평균 -> 정규분포
np.random.seed(42)
sample_sizes = [1, 2, 5, 30]
n_experiments = 10000

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, n in zip(axes, sample_sizes):
    means = []
    for _ in range(n_experiments):
        sample = np.random.randint(1, 7, size=n)
        means.append(np.mean(sample))
    means = np.array(means)

    ax.hist(means, bins=50, density=True, alpha=0.7, color='steelblue')
    ax.set_title(f'n = {n}\nstd = {np.std(means):.3f}')
    ax.set_xlim(0, 7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Central Limit Theorem: Sample Mean of Dice Rolls', y=1.02)
plt.tight_layout()
plt.show()

print("n이 커질수록 표본 평균의 분포가 정규분포에 가까워진다!")

---
## 3. 베이즈 정리 (Bayes' Theorem)

새로운 증거(데이터)를 관찰한 후 기존 믿음(확률)을 업데이트하는 공식.

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

| 용어 | 의미 | 예시 |
|------|------|------|
| $P(A)$ | 사전 확률 (Prior) | 스팸 메일일 확률 |
| $P(B|A)$ | 우도 (Likelihood) | 스팸일 때 "할인"이 포함될 확률 |
| $P(A|B)$ | 사후 확률 (Posterior) | "할인"이 있을 때 스팸일 확률 |
| $P(B)$ | 증거 (Evidence) | "할인"이 나타날 전체 확률 |

**ML에서**: 나이브 베이즈 분류기, 베이지안 추론, 사후 확률 최대화

In [ ]:
# 예제: 스팸 필터
# 이메일에 "할인"이라는 단어가 있을 때, 스팸일 확률은?

# 사전 확률
P_spam = 0.3          # 전체 메일 중 스팸 비율: 30%
P_not_spam = 0.7

# 우도 (Likelihood)
P_discount_given_spam = 0.8      # 스팸 메일에 "할인"이 포함될 확률: 80%
P_discount_given_not_spam = 0.1  # 정상 메일에 "할인"이 포함될 확률: 10%

# 증거: P("할인") = P("할인"|스팸)P(스팸) + P("할인"|정상)P(정상)
P_discount = P_discount_given_spam * P_spam + P_discount_given_not_spam * P_not_spam
print(f"P(할인) = {P_discount_given_spam}*{P_spam} + {P_discount_given_not_spam}*{P_not_spam} = {P_discount}")

# 베이즈 정리 적용
P_spam_given_discount = (P_discount_given_spam * P_spam) / P_discount

print(f"\nP(스팸 | 할인) = {P_spam_given_discount:.4f}")
print(f"-> '할인'이 포함된 메일은 {P_spam_given_discount*100:.1f}% 확률로 스팸!")
print(f"-> 사전 확률 {P_spam*100}%에서 {P_spam_given_discount*100:.1f}%로 업데이트됨")

In [ ]:
# 베이즈 정리 직관적 시각화: 면적 다이어그램
fig, ax = plt.subplots(figsize=(10, 6))

# 전체 = 1.0 (정규화된 영역)
# 스팸 영역
ax.barh(0, P_spam, height=0.8, color='salmon', alpha=0.8, label=f'Spam (P={P_spam})')
ax.barh(0, P_not_spam, left=P_spam, height=0.8, color='lightblue', alpha=0.8, label=f'Not Spam (P={P_not_spam})')

# "할인" 포함 영역 (빗금)
spam_discount = P_spam * P_discount_given_spam
notspam_discount = P_not_spam * P_discount_given_not_spam

ax.barh(-1, spam_discount, height=0.8, color='red', alpha=0.6,
        label=f'Spam AND "할인" ({spam_discount:.2f})')
ax.barh(-1, notspam_discount, left=spam_discount, height=0.8, color='blue', alpha=0.4,
        label=f'Not Spam AND "할인" ({notspam_discount:.2f})')

ax.set_yticks([0, -1])
ax.set_yticklabels(['All emails', 'Contains "할인"'])
ax.set_xlabel('Probability')
ax.set_title('Bayes Theorem Visualization\n'
             f'P(Spam|"할인") = {spam_discount:.2f} / ({spam_discount:.2f}+{notspam_discount:.2f}) = {P_spam_given_discount:.2f}')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# 베이즈 업데이트 시각화: 데이터가 쌓일수록 사후 확률이 수렴
np.random.seed(42)

# 실제 동전: p=0.7 (앞면 확률)
true_p = 0.7
n_flips = 100
flips = np.random.binomial(1, true_p, size=n_flips)

# 베이즈 업데이트: Beta 분포 사용
# Prior: Beta(1, 1) = Uniform(0, 1) -> "아무것도 모르는 상태"
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
x = np.linspace(0, 1, 200)
checkpoints = [0, 1, 5, 10, 30, 100]

for ax, n in zip(axes.flat, checkpoints):
    a = 1 + np.sum(flips[:n])       # alpha: 1 + 앞면 수
    b = 1 + n - np.sum(flips[:n])   # beta: 1 + 뒷면 수
    y = stats.beta.pdf(x, a, b)
    ax.plot(x, y, 'b-', linewidth=2)
    ax.fill_between(x, y, alpha=0.2)
    ax.axvline(true_p, color='red', linestyle='--', label=f'True p={true_p}')
    mean = a / (a + b)
    ax.axvline(mean, color='green', linestyle=':', label=f'Estimate={mean:.3f}')
    if n > 0:
        ax.set_title(f'After {n} flips (H={np.sum(flips[:n])})')
    else:
        ax.set_title('Prior (no data)')
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Bayesian Update: Estimating Coin Bias', fontsize=14)
plt.tight_layout()
plt.show()

print("데이터가 쌓일수록 추정치(초록)가 실제값(빨간)에 수렴한다!")

---
## 4. 최대우도추정 (Maximum Likelihood Estimation, MLE)

관측된 데이터가 나올 확률(우도)을 **최대화**하는 파라미터를 찾는 방법.

$$\hat{\theta}_{MLE} = \arg\max_\theta P(\text{data} | \theta)$$

### 직관적 설명

동전을 10번 던져서 앞면 7번, 뒷면 3번이 나왔다고 하자.
- 만약 $p=0.5$면: $P(\text{data}) = \binom{10}{7} \cdot 0.5^{7} \cdot 0.5^{3} \approx 0.117$
- 만약 $p=0.7$면: $P(\text{data}) = \binom{10}{7} \cdot 0.7^{7} \cdot 0.3^{3} \approx 0.267$
- $p=0.7$일 때 데이터가 나올 확률이 더 높다! $\hat{p}_{MLE} = 7/10 = 0.7$

**ML에서**: 신경망 학습 = 결국 MLE. Cross-entropy 손실 최소화 = 음의 로그 우도 최소화.

In [ ]:
# 동전 던지기 MLE: 직접 구현
np.random.seed(42)

# 데이터 생성: 실제 p=0.7인 동전을 50번 던짐
true_p = 0.7
n = 50
data = np.random.binomial(1, true_p, size=n)
n_heads = np.sum(data)
n_tails = n - n_heads
print(f"데이터: {n}번 중 앞면 {n_heads}번, 뒷면 {n_tails}번")

# Log-Likelihood 함수
def log_likelihood(p, n_heads, n_tails):
    if p <= 0 or p >= 1:
        return -np.inf
    return n_heads * np.log(p) + n_tails * np.log(1 - p)

# 다양한 p에 대해 우도 계산
p_values = np.linspace(0.01, 0.99, 200)
ll_values = [log_likelihood(p, n_heads, n_tails) for p in p_values]

# MLE: 해석적 해 = n_heads / n
p_mle = n_heads / n
print(f"MLE 추정치: p_hat = {n_heads}/{n} = {p_mle}")
print(f"실제값: p = {true_p}")

In [ ]:
# MLE 시각화
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(p_values, ll_values, 'b-', linewidth=2)
ax.axvline(p_mle, color='red', linestyle='--', linewidth=2,
           label=f'MLE: p_hat = {p_mle:.2f}')
ax.axvline(true_p, color='green', linestyle=':', linewidth=2,
           label=f'True: p = {true_p}')
ax.set_xlabel('p (probability of heads)')
ax.set_ylabel('Log-Likelihood')
ax.set_title(f'MLE for Coin Flip (n={n}, heads={n_heads})')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 정규분포의 MLE: 평균과 분산 추정
np.random.seed(42)

# 실제 파라미터
true_mu = 5.0
true_sigma = 2.0

# 데이터 생성
data = np.random.normal(true_mu, true_sigma, size=100)

# MLE 추정 (정규분포의 MLE = 표본 평균, 표본 표준편차)
mu_mle = np.mean(data)
sigma_mle = np.std(data)  # MLE는 n으로 나눔 (편향 추정)

print(f"실제 평균: {true_mu}, MLE 추정: {mu_mle:.4f}")
print(f"실제 표준편차: {true_sigma}, MLE 추정: {sigma_mle:.4f}")

# 시각화
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(-2, 12, 200)
ax.hist(data, bins=20, density=True, alpha=0.5, color='steelblue', label='Data')
ax.plot(x, stats.norm.pdf(x, true_mu, true_sigma), 'g-', linewidth=2,
        label=f'True: N({true_mu}, {true_sigma}^2)')
ax.plot(x, stats.norm.pdf(x, mu_mle, sigma_mle), 'r--', linewidth=2,
        label=f'MLE: N({mu_mle:.2f}, {sigma_mle:.2f}^2)')
ax.legend(fontsize=11)
ax.set_title('MLE for Normal Distribution')
ax.grid(True, alpha=0.3)
plt.show()

### MLE 해석적 풀이 (참고)

동전 던지기의 경우, 로그 우도를 미분하면:

$$\ell(p) = k \log p + (n-k) \log(1-p)$$

$$\frac{d\ell}{dp} = \frac{k}{p} - \frac{n-k}{1-p} = 0$$

$$\hat{p}_{MLE} = \frac{k}{n}$$

직관과 일치: 성공 횟수 / 전체 시행 수

---
## 5. 정보이론 (Information Theory)

### 5.1 엔트로피 (Entropy)

확률분포의 **불확실성(정보량)**을 측정.

$$H(P) = -\sum_{x} P(x) \log P(x)$$

- 모든 결과가 동일하게 가능 -> 엔트로피 최대 (불확실성 높음)
- 한 결과가 확실 -> 엔트로피 = 0 (불확실성 없음)

**ML에서**: 의사결정 나무의 분할 기준, 정보 이득(Information Gain)

In [ ]:
def entropy(probs):
    """확률분포의 엔트로피 계산 (자연로그, 단위: nats)"""
    probs = np.array(probs)
    # 0인 항은 제외 (0 * log(0) = 0으로 처리)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log(probs))

def entropy_bits(probs):
    """확률분포의 엔트로피 계산 (log base 2, 단위: bits)"""
    probs = np.array(probs)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

# 다양한 동전의 엔트로피
print("=== 동전 엔트로피 (bits) ===")
print(f"공정한 동전 [0.5, 0.5]: H = {entropy_bits([0.5, 0.5]):.4f} bits")
print(f"편향 동전 [0.8, 0.2]:   H = {entropy_bits([0.8, 0.2]):.4f} bits")
print(f"편향 동전 [0.99, 0.01]: H = {entropy_bits([0.99, 0.01]):.4f} bits")
print(f"확실한 동전 [1.0, 0.0]: H = {entropy_bits([1.0, 0.0]):.4f} bits")

# 주사위 엔트로피
print(f"\n=== 주사위 엔트로피 (bits) ===")
print(f"공정한 주사위: H = {entropy_bits([1/6]*6):.4f} bits")
print(f"편향 주사위 [0.5, 0.1, 0.1, 0.1, 0.1, 0.1]: H = {entropy_bits([0.5, 0.1, 0.1, 0.1, 0.1, 0.1]):.4f} bits")

In [ ]:
# 이진 엔트로피 함수 시각화
p_values = np.linspace(0.001, 0.999, 200)
h_values = [-p * np.log2(p) - (1-p) * np.log2(1-p) for p in p_values]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_values, h_values, 'b-', linewidth=2)
ax.set_xlabel('P(Head)')
ax.set_ylabel('Entropy (bits)')
ax.set_title('Binary Entropy Function H(p)')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='p=0.5 (max entropy)')
ax.annotate('Maximum\nuncertainty', xy=(0.5, 1.0), fontsize=11,
            ha='center', color='red')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("p=0.5일 때 엔트로피 최대 = 가장 예측하기 어려움")
print("p=0 또는 p=1일 때 엔트로피 0 = 결과가 확실함")

### 5.2 Cross-Entropy

실제 분포 P에서 생성된 데이터를 예측 분포 Q로 인코딩할 때 필요한 평균 비트 수.

$$H(P, Q) = -\sum_{x} P(x) \log Q(x)$$

**ML에서의 핵심 역할**: 분류 모델의 손실 함수!

- 실제 레이블 P = [0, 1, 0] (정답: 클래스 2)
- 모델 예측 Q = [0.1, 0.8, 0.1]
- Cross-entropy = $-\log(0.8) \approx 0.223$ (낮을수록 좋음)

항상 $H(P, Q) \geq H(P)$이고, $P = Q$일 때만 등호 성립.

In [ ]:
def cross_entropy(p, q):
    """Cross-entropy H(P, Q): P가 실제, Q가 예측 (자연로그)"""
    p = np.array(p)
    q = np.array(q)
    # P(x)=0인 항은 기여하지 않음
    mask = p > 0
    return -np.sum(p[mask] * np.log(q[mask]))

# 분류 예제: 3개 클래스, 정답은 클래스 2 (인덱스 1)
true_label = [0, 1, 0]  # one-hot encoding

# 다양한 예측의 cross-entropy
predictions = {
    'Perfect':     [0.001, 0.998, 0.001],
    'Good':        [0.05, 0.90, 0.05],
    'Okay':        [0.10, 0.70, 0.20],
    'Bad':         [0.30, 0.40, 0.30],
    'Very Bad':    [0.60, 0.10, 0.30],
}

print("Cross-Entropy for different predictions:")
print(f"True label: {true_label}\n")
for name, q in predictions.items():
    ce = cross_entropy(true_label, q)
    print(f"{name:10s}  Q={q}  ->  CE = {ce:.4f}")

print(f"\n-> 예측이 정확할수록 Cross-Entropy가 낮다 (좋은 모델)")

In [ ]:
# Cross-Entropy 시각화: 예측 확률에 따른 손실
p_pred = np.linspace(0.01, 0.99, 200)
loss = -np.log(p_pred)  # 정답 클래스에 대한 예측 확률

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_pred, loss, 'b-', linewidth=2)
ax.set_xlabel('Predicted probability for correct class')
ax.set_ylabel('Cross-Entropy Loss (-log(p))')
ax.set_title('Cross-Entropy Loss vs Prediction Confidence')

# 주요 포인트 표시
for p, name in [(0.9, 'Good'), (0.5, 'Uncertain'), (0.1, 'Wrong')]:
    ax.plot(p, -np.log(p), 'ro', markersize=8)
    ax.annotate(f'{name}\np={p}, loss={-np.log(p):.2f}',
                xy=(p, -np.log(p)), xytext=(p+0.05, -np.log(p)+0.3),
                fontsize=9, arrowprops=dict(arrowstyle='->', color='red'))

ax.grid(True, alpha=0.3)
plt.show()

print("예측 확률이 낮을수록 (틀릴수록) loss가 급격히 증가!")
print("-> 모델이 '자신있게 틀리면' 큰 패널티를 받는다")

### 5.3 KL Divergence (Kullback-Leibler Divergence)

두 확률분포 P와 Q의 **차이**를 측정.

$$D_{KL}(P \| Q) = \sum_{x} P(x) \log \frac{P(x)}{Q(x)} = H(P, Q) - H(P)$$

- $D_{KL} \geq 0$ (항상 0 이상)
- $D_{KL} = 0 \iff P = Q$ (두 분포가 같으면 0)
- **비대칭**: $D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$

**ML에서의 활용**:
- VAE의 손실 함수에 KL divergence 항 포함
- 지식 증류(Knowledge Distillation)에서 Teacher-Student 분포 차이 측정
- Cross-entropy 최소화 = KL divergence 최소화 (H(P)는 상수이므로)

In [ ]:
def kl_divergence(p, q):
    """KL Divergence D_KL(P || Q)"""
    p = np.array(p)
    q = np.array(q)
    mask = p > 0
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

# 두 분포의 KL Divergence
P = [0.4, 0.3, 0.2, 0.1]       # 실제 분포
Q1 = [0.35, 0.30, 0.20, 0.15]  # 비슷한 분포
Q2 = [0.1, 0.1, 0.1, 0.7]      # 매우 다른 분포
Q3 = [0.25, 0.25, 0.25, 0.25]  # 균일 분포

print(f"P  = {P}")
print(f"Q1 = {Q1}  (비슷)")
print(f"Q2 = {Q2}  (매우 다름)")
print(f"Q3 = {Q3}  (균일)")

print(f"\nD_KL(P || Q1) = {kl_divergence(P, Q1):.4f}  (비슷 -> 작은 값)")
print(f"D_KL(P || Q2) = {kl_divergence(P, Q2):.4f}  (다름 -> 큰 값)")
print(f"D_KL(P || Q3) = {kl_divergence(P, Q3):.4f}")

# 비대칭성 확인
print(f"\n비대칭성 확인:")
print(f"D_KL(P || Q2) = {kl_divergence(P, Q2):.4f}")
print(f"D_KL(Q2 || P) = {kl_divergence(Q2, P):.4f}")
print(f"-> KL divergence는 대칭이 아니다!")

In [ ]:
# Cross-Entropy = Entropy + KL Divergence 관계 확인
P = [0.4, 0.3, 0.2, 0.1]
Q = [0.35, 0.30, 0.20, 0.15]

h_p = entropy(P)             # H(P) in nats
ce = cross_entropy(P, Q)     # H(P, Q) in nats
kl = kl_divergence(P, Q)     # D_KL(P||Q) in nats

print(f"H(P)       = {h_p:.4f} nats")
print(f"D_KL(P||Q) = {kl:.4f} nats")
print(f"H(P) + KL  = {h_p + kl:.4f} nats")
print(f"H(P, Q)    = {ce:.4f} nats")
print(f"\n-> H(P, Q) = H(P) + D_KL(P||Q) 확인!")

In [ ]:
# 시각화: 두 정규분포의 KL Divergence
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.linspace(-6, 10, 300)

cases = [
    ('Nearly identical', 0, 1, 0.2, 1),
    ('Different means', 0, 1, 3, 1),
    ('Different variances', 0, 1, 0, 2.5),
]

for ax, (title, mu_p, sig_p, mu_q, sig_q) in zip(axes, cases):
    p_pdf = stats.norm.pdf(x, mu_p, sig_p)
    q_pdf = stats.norm.pdf(x, mu_q, sig_q)

    # 정규분포 KL divergence 해석해
    kl_val = np.log(sig_q / sig_p) + (sig_p**2 + (mu_p - mu_q)**2) / (2 * sig_q**2) - 0.5

    ax.plot(x, p_pdf, 'b-', linewidth=2, label=f'P: N({mu_p},{sig_p}^2)')
    ax.plot(x, q_pdf, 'r--', linewidth=2, label=f'Q: N({mu_q},{sig_q}^2)')
    ax.fill_between(x, p_pdf, alpha=0.15, color='blue')
    ax.fill_between(x, q_pdf, alpha=0.15, color='red')
    ax.set_title(f'{title}\nKL(P||Q) = {kl_val:.4f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.4 Cross-Entropy Loss가 ML 학습에서 쓰이는 이유

정리하면:

1. **MLE로 학습** = 데이터의 우도를 최대화
2. 우도 최대화 = **음의 로그 우도(NLL) 최소화**
3. NLL = **Cross-Entropy** (분류의 경우)
4. Cross-entropy 최소화 = **KL divergence 최소화** ($H(P)$는 상수)

$$\text{모델의 예측 분포 Q를 실제 분포 P에 가깝게 만드는 것!}$$

In [ ]:
# NLL과 Cross-Entropy가 같음을 보이는 예제
# 3-class 분류, 5개 데이터
np.random.seed(42)

# 정답 레이블
y_true = np.array([0, 1, 2, 1, 0])
# 모델 예측 (softmax 출력)
y_pred = np.array([
    [0.7, 0.2, 0.1],
    [0.1, 0.8, 0.1],
    [0.2, 0.3, 0.5],
    [0.1, 0.6, 0.3],
    [0.9, 0.05, 0.05],
])

# 방법 1: NLL 직접 계산
nll = 0
for i in range(len(y_true)):
    nll += -np.log(y_pred[i, y_true[i]])
nll /= len(y_true)

# 방법 2: Cross-entropy 계산
ce = 0
for i in range(len(y_true)):
    one_hot = np.zeros(3)
    one_hot[y_true[i]] = 1
    ce += cross_entropy(one_hot, y_pred[i])
ce /= len(y_true)

print(f"NLL (평균):            {nll:.4f}")
print(f"Cross-Entropy (평균):  {ce:.4f}")
print(f"같은가? {np.isclose(nll, ce)}")
print(f"\n-> NLL = Cross-Entropy Loss!")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 몬티홀 문제 시뮬레이션

3개의 문 뒤에 자동차 1대, 염소 2마리가 있다.
1. 참가자가 문 하나를 선택
2. 진행자가 염소가 있는 문 하나를 열어줌
3. 참가자에게 선택을 바꿀 기회를 줌

**문제**: 바꾸는 것이 유리한가? 시뮬레이션으로 확인하세요.
- 기대 결과: 바꾸면 2/3, 안 바꾸면 1/3

In [ ]:
# TODO: 몬티홀 문제 시뮬레이션
# 1. n_simulations = 10000 으로 설정
# 2. 각 시뮬레이션에서:
#    - 자동차 위치를 랜덤 선택 (0, 1, 2)
#    - 참가자 선택을 랜덤 선택 (0, 1, 2)
#    - "바꾸지 않는" 전략: 원래 선택 == 자동차 위치면 승리
#    - "바꾸는" 전략: 원래 선택 != 자동차 위치면 승리 (왜 그런지 생각해보기!)
# 3. 두 전략의 승률을 출력하고 비교


### 연습 2: 나이브 베이즈 분류기 구현

아래 데이터로 간단한 나이브 베이즈 스팸 필터를 구현하세요.

In [ ]:
# 학습 데이터
emails = [
    ("할인 특가 세일 지금 구매", "spam"),
    ("무료 경품 당첨 축하", "spam"),
    ("긴급 할인 마감 임박", "spam"),
    ("내일 회의 시간 변경", "ham"),
    ("프로젝트 보고서 검토 부탁", "ham"),
    ("점심 메뉴 추천 부탁", "ham"),
]

test_email = "무료 할인 이벤트"

# TODO: 나이브 베이즈 분류기 구현
# 1. P(spam), P(ham) 사전 확률 계산
# 2. 각 단어의 조건부 확률 P(word|spam), P(word|ham) 계산
#    (Laplace smoothing 적용: 분자에 +1, 분모에 +전체 고유 단어 수)
# 3. test_email에 대해 P(spam|test_email)과 P(ham|test_email) 비교
#    (log 확률로 계산하면 underflow 방지)
# 4. 결과 출력: spam인지 ham인지


### 연습 3: Softmax와 Cross-Entropy Loss 구현

신경망 출력(logits)에서 Cross-Entropy Loss를 직접 계산하세요.

In [ ]:
# 4-class 분류 문제
logits = np.array([2.0, 1.0, 0.5, -1.0])  # 신경망의 raw output
true_class = 0  # 정답 클래스

# TODO: 
# 1. softmax 함수 구현: softmax(z)_i = exp(z_i) / sum(exp(z_j))
#    (수치 안정성을 위해 z - max(z) 적용)
# 2. softmax를 logits에 적용하여 확률 분포 얻기
# 3. 확률 분포의 합이 1인지 확인
# 4. Cross-entropy loss 계산: -log(softmax[true_class])
# 5. 결과 출력


---
## 핵심 정리

| 개념 | ML에서의 역할 |
|------|---------------|
| 조건부 확률 | 분류 모델 = P(Y|X) 학습 |
| 베이즈 정리 | 사후 확률 업데이트, 나이브 베이즈 분류기 |
| 베르누이/이항분포 | 이진 분류, 성공/실패 모델링 |
| 정규분포 | 가중치 초기화, VAE, 중심극한정리 |
| MLE | 모델 학습의 이론적 기반 (손실 최소화 = NLL 최소화) |
| 엔트로피 | 불확실성 측정, 의사결정 나무 |
| Cross-Entropy | 분류 모델의 손실 함수 |
| KL Divergence | VAE 손실, 지식 증류, 분포 차이 측정 |

**핵심 연결**: Cross-entropy loss 최소화 = MLE = 예측 분포를 실제 분포에 맞추기

**다음 노트북**: [03-calculus-for-ml.ipynb](03-calculus-for-ml.ipynb) - ML을 위한 미적분